# Tutorial: PM-01 Naive Bayes

Audience:
- Learners studying probabilistic classification and conditional independence assumptions.

Prerequisites:
- Probability basics, Bayes' rule, conditional probability, and train/test evaluation.

Learning goals:
- Implement Bernoulli naive Bayes from scratch with Laplace smoothing.
- Fit and evaluate the classifier on a synthetic binary-feature dataset.
- Inspect posterior log-odds contributions from individual features.


## Outline

1. Generate a labeled binary-feature dataset.
2. Estimate class priors and class-conditional Bernoulli probabilities.
3. Compute posterior scores and predictions.
4. Evaluate accuracy and inspect feature contributions.


In [ ]:
from __future__ import annotations

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

SEED = 5
rng = np.random.default_rng(SEED)
SEED


## Step 1 - Build a binary-feature classification problem

Each class has its own Bernoulli feature probabilities. This matches the modeling assumptions of Bernoulli naive Bayes, so the notebook can focus on the mechanics of inference and evaluation.


In [ ]:
def make_binary_dataset(n_samples: int = 800, seed: int = SEED):
    local_rng = np.random.default_rng(seed)
    y = local_rng.binomial(1, 0.52, size=n_samples)
    probs_y0 = np.array([0.15, 0.20, 0.75, 0.70, 0.30, 0.25])
    probs_y1 = np.array([0.80, 0.70, 0.25, 0.35, 0.75, 0.65])
    probs = np.where(y[:, None] == 1, probs_y1, probs_y0)
    x = local_rng.binomial(1, probs).astype(np.float64)
    feature_names = [
        "token_offer",
        "token_bonus",
        "token_meeting",
        "token_project",
        "token_click",
        "token_discount",
    ]
    return x, y.astype(int), feature_names


X, y, feature_names = make_binary_dataset()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

{"train_shape": X_train.shape, "test_shape": X_test.shape, "class_balance_train": y_train.mean()}


## Step 2 - Implement Bernoulli naive Bayes

With conditional independence, the class-conditional likelihood factorizes across features. Laplace smoothing prevents zero probabilities from destroying the posterior score.


In [ ]:
class BernoulliNaiveBayes:
    def __init__(self, alpha: float = 1.0):
        self.alpha = alpha

    def fit(self, X: np.ndarray, y: np.ndarray) -> "BernoulliNaiveBayes":
        self.classes_, counts = np.unique(y, return_counts=True)
        self.class_log_prior_ = np.log(counts / counts.sum())
        feature_probs = []
        for cls in self.classes_:
            X_cls = X[y == cls]
            smoothed = (X_cls.sum(axis=0) + self.alpha) / (X_cls.shape[0] + 2.0 * self.alpha)
            feature_probs.append(smoothed)
        self.feature_prob_ = np.vstack(feature_probs)
        return self

    def predict_log_proba(self, X: np.ndarray) -> np.ndarray:
        log_prob = np.log(self.feature_prob_)
        log_inv_prob = np.log(1.0 - self.feature_prob_)
        joint = X @ log_prob.T + (1.0 - X) @ log_inv_prob.T + self.class_log_prior_
        normalizer = np.logaddexp.reduce(joint, axis=1, keepdims=True)
        return joint - normalizer

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return np.exp(self.predict_log_proba(X))

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.classes_[np.argmax(self.predict_log_proba(X), axis=1)]


model = BernoulliNaiveBayes(alpha=1.0).fit(X_train, y_train)
np.round(model.feature_prob_, 3)


## Step 3 - Evaluate predictive performance

The model is simple, but because the dataset roughly respects the model assumptions, it should perform well. The confusion matrix helps show where remaining mistakes occur.


In [ ]:
test_proba = model.predict_proba(X_test)
test_pred = model.predict(X_test)
acc = accuracy_score(y_test, test_pred)
cm = confusion_matrix(y_test, test_pred)

fig, ax = plt.subplots(figsize=(4.6, 3.6))
im = ax.imshow(cm, cmap="Blues")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["pred 0", "pred 1"])
ax.set_yticklabels(["true 0", "true 1"])
ax.set_title("Naive Bayes confusion matrix")
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
plt.close(fig)

{"accuracy": round(float(acc), 4), "confusion_matrix": cm.tolist()}


## Step 4 - Inspect posterior log-odds contributions

For binary classes, the posterior score is additive in the feature contributions. This makes naive Bayes a useful teaching model even when its independence assumption is unrealistic.


In [ ]:
log_odds_present = np.log(model.feature_prob_[1] / model.feature_prob_[0])
log_odds_absent = np.log((1.0 - model.feature_prob_[1]) / (1.0 - model.feature_prob_[0]))

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.8), sharey=True)
axes[0].bar(feature_names, log_odds_present, color="tab:blue")
axes[0].set_title("Contribution when feature = 1")
axes[0].set_ylabel("log-odds contribution toward class 1")
axes[0].tick_params(axis="x", rotation=35)
axes[1].bar(feature_names, log_odds_absent, color="tab:orange")
axes[1].set_title("Contribution when feature = 0")
axes[1].tick_params(axis="x", rotation=35)
fig.tight_layout()
plt.close(fig)

{
    "feature_prob_class_0": dict(zip(feature_names, np.round(model.feature_prob_[0], 3))),
    "feature_prob_class_1": dict(zip(feature_names, np.round(model.feature_prob_[1], 3))),
}


## Summary

This notebook implemented Bernoulli naive Bayes directly from the model assumptions, estimated its parameters from counts, and evaluated it on held-out data. The classifier works by adding log-probability contributions from features under a conditional-independence assumption, which makes it both computationally cheap and easy to interpret.
